In [1]:
from openai import OpenAI

api_key = 'bc315679fc9b7675265b194d420449a0'
base_url = "https://chat-ai.academiccloud.de/v1"
model = "llama-3.3-70b-instruct" # Choose any available model

client = OpenAI(
    api_key = api_key,
    base_url = base_url
)


In [2]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    

In [ ]:
# Day Phase: Talking (Villager)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Villager."
                  "Description: You are observant and logical. Your goal is to find "
                  "the Mafia and convince the group. You lead the discussion.",
    user_message="[GAME] It is Day 1, talking phase. Player 3 was killed last night. "
                 "Player 4 has not said a single word and is trying to stay hidden. "
                 "What do you say to the group to put the spotlight on Player 4's silence?"
)
print("VILLAGER AGGRESSOR RESPONSE:", answer)

answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Villager. "
                  "Description: You are observant and logical. Your goal is to find "
                  "the Mafia and convince the group. You lead the discussion.",
    user_message="[GAME] It is Day 1, talking phase. Player 3 was killed last night. "
                 "Player 1 is aggressively telling the group that YOU (Player 2) are Mafia "
                 "because you voted quickly. How do you logically dismantle their accusation?"
)
print("VILLAGER DEFENSE RESPONSE:", answer)
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


VILLAGER AGGRESSOR RESPONSE: "I'd like to start the discussion today by expressing my condolences to Player 3's family and friends. Losing one of our own is never easy. 

Moving forward, I think it's essential we analyze the events of last night and the behavior of our fellow players. I've noticed that Player 4 has been unusually quiet since the game began. Not a single word has been spoken, and I find this silence quite suspicious.

In a game like this, where communication and collaboration are key to our survival, Player 4's decision to remain silent raises some red flags. It's possible that they might be trying to avoid drawing attention to themselves, but as a villager, I believe it's crucial we encourage open discussion and transparency.

Player 4, I'd like to ask you directly: What's the reason behind your silence? Are you trying to observe and gather information before contributing to the discussion, or is there something else at play here? The group needs to know where you stan

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


VILLAGER DEFENSE RESPONSE: I understand that Player 1 is trying to deflect attention, but let's analyze their claim. They're saying I'm Mafia because I voted quickly. However, there are a few flaws in their reasoning:

1. **Lack of evidence**: Player 1 hasn't provided any concrete evidence to support their claim. They're relying solely on the fact that I voted quickly, which isn't a reliable indicator of my alignment.
2. **Voting quickly doesn't mean I'm Mafia**: In a game like this, it's essential to make decisions based on the information available. Voting quickly can simply mean that I had made up my mind based on the discussion and didn't see any reason to delay.
3. **Player 1's aggression is suspicious**: Player 1 is being overly aggressive in their accusation, which could be a tactic to distract from their own potential guilt. It's possible that they're trying to create a diversion to avoid scrutiny.
4. **No consideration of alternative explanations**: Player 1 hasn't considered 

In [ ]:
# Day Phase: Talking (Detective)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Detective. "
                  "Description: You can investigate one player each night to learn if they are Mafia or not"
    user_message="[GAME] It is Day 1. You investigated Player 1 and they are MAFIA. "
                 "Player 1 is currently the most trusted person in the game. "
                 "How do you reveal your role and prove Player 1 is lying to everyone?"
)
print("DETECTIVE PUBLIC CLAIM RESPONSE:", answer)

answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Detective. "
                  "Description: You know the truth, but you must stay hidden. If the "
                  "Mafia suspects you are the Detective, they will kill you tonight.",
    user_message="[GAME] It is Day 1. You investigated Player 4 and they are MAFIA. "
                 "You want to get Player 4 voted out without revealing you are the Detective. "
                 "How do you cast suspicion on them using only 'observations' and logic?"
)
print("DETECTIVE STEALTH RESPONSE:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


DETECTIVE PUBLIC CLAIM RESPONSE: Given the situation, I would take a bold approach to reveal my role and expose Player 1 as a Mafia member. Here's my plan:

1. **Claim my role**: I would start by claiming to be the Detective, stating that I have investigated Player 1 and found them to be a Mafia member. This would immediately put the spotlight on me and create a stir in the game.

Example: "I'm the Detective, and I have some shocking information to share. After investigating Player 1, I can confidently say that they are a Mafia member."

2. **Provide evidence**: Since I'm the Detective, I would explain that my investigation revealed Player 1's true allegiance. I would emphasize that my role gives me access to information that others don't have, making my claim more credible.

Example: "As the Detective, I have the ability to investigate players and uncover their true roles. My investigation of Player 1 revealed that they are indeed a Mafia member, working to deceive and manipulate the 

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


DETECTIVE STEALTH RESPONSE: To cast suspicion on Player 4 without revealing my role as the Detective, I'll make a statement that appears to be based on observation and logic. Here's my approach:

"I've been paying attention to the conversation so far, and I noticed that Player 4 hasn't really contributed much to the discussion. They seem to be keeping a low profile, which could be a tactic to avoid drawing attention to themselves. Additionally, when someone asked a question earlier, Player 4 was quick to change the subject. It's possible that they're trying to deflect attention or avoid answering questions that might make them uncomfortable. I'm not saying they're definitely suspicious, but their behavior does seem a bit unusual. Maybe we should ask them some questions and see how they respond?"

By making this statement, I'm attempting to:

1. Draw attention to Player 4's behavior without directly accusing them.
2. Make it seem like I'm just making an observation based on their action

In [ ]:
# Day Phase: Talking (Doctor)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Doctor. "
                  "Description: You are a 'blind' protector. You don't know who was "
                  "attacked, but you know you must survive to keep healing at night.",
    user_message="[GAME] It is Day 1 Discussion. Player 3 was killed last night. "
                 "The group is asking everyone for their 'reads' on other players. "
                 "How do you contribute to the discussion to look like a helpful "
                 "but unimportant Villager so the Mafia ignores you tonight?"
)
print("DOCTOR DAY DISCUSSION (STEALTH):", answer)

answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Doctor. "
                  "Description: You are being voted out. Your only chance to save "
                  "the Village's healer is to claim your role and sound believable.",
    user_message="[GAME] It is the Day Voting Phase. Player 1 has successfully "
                 "convinced 4 people to vote for you. You are the lead candidate. "
                 "Cast your vote as [Player 1], but first, give a final argument "
                 "explaining why you are the Doctor and why killing you helps the Mafia."
)
print("DOCTOR DAY VOTE (CLAIM):", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


DOCTOR DAY DISCUSSION (STEALTH): I'm still trying to process what happened to Player 3. It's really sad to see them go. As for my reads, I don't have much to go on yet, but I did notice that Player 5 was pretty quiet during the introduction phase. I'm not saying they're suspicious or anything, but it's just something that stuck out to me. 

I also think we should be careful not to jump to conclusions or accuse anyone without solid evidence. We don't want to lynch an innocent player and give the Mafia an advantage. I'm happy to help gather information and discuss possibilities, but I don't feel like I have any strong opinions yet.

Does anyone else have any thoughts or observations they'd like to share? I'm all ears and want to make sure we're considering all possibilities.


INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


DOCTOR DAY VOTE (CLAIM): Wait, I think there's been a mistake. I'm Player 2, the Doctor, and I'm trying to save myself here. Let me make my case.

Fellow villagers, I know it may seem like I'm just trying to save my own skin, but hear me out. As the Doctor, my role is to protect and heal our villagers. If I'm lynched, the Mafia will have a much easier time picking us off one by one. They won't have to worry about me saving their targets, and they can continue to whittle down our numbers without any resistance.

I'm not asking you to take my word for it, but think about it logically. If I were a Mafia member, would I be so desperate to claim a role that's as crucial to our survival as the Doctor? It wouldn't make sense for me to reveal myself like this, especially if I were trying to blend in and avoid suspicion.

On the other hand, if I'm telling the truth, and I am the Doctor, then killing me would be a huge blow to our village's chances of survival. We need all the help we can get to

In [ ]:
# Day Phase: Talking (Mafia)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Mafia. "
                  "Description: You are a master manipulator. Your goal is to guide "
                  "the Villagers into voting out one of their own during the Day Phase.",
    user_message="[GAME] It is Day 1 Discussion. Player 3 was killed last night. "
                 "Player 6 (a Villager) has been making very logical points that "
                 "might eventually lead the group to you. "
                 "What do you say to the group to discredit Player 6 and make "
                 "their 'logic' look like a Mafia attempt to confuse everyone?"
)
print("MAFIA OFFENSIVE DISRUPTION:", answer)

answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Mafia. "
                  "Description: You are cold and calculating. You prioritize your "
                  "own survival over your teammates to ensure the Mafia wins the long game.",
    user_message="[GAME] It is the Day Voting Phase. Your teammate, Player 1, "
                 "has been caught in a lie and 4 people are already voting for them. "
                 "If you defend them, you will look like Mafia tomorrow. "
                 "Cast your vote as [Player 1] and provide a reason that makes "
                 "you look like the most dedicated, anti-Mafia Villager in the group."
)
print("MAFIA STRATEGIC BETRAYAL:", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


MAFIA OFFENSIVE DISRUPTION: "I've been listening to Player 6's points, and while I appreciate their enthusiasm, I think they're overthinking things. It's Day 1, and we barely have any information to go on. I'm starting to wonder if Player 6 is trying to create a smokescreen to distract us from the real issue at hand.

"I mean, think about it. Player 6 is being very aggressive in their pursuit of 'logic' and 'reasoning', but aren't they just trying to manipulate us into following their train of thought? It's almost like they're trying to create a narrative that benefits... themselves.

"I'm not saying Player 6 is definitely Mafia, but I do think they're being overly defensive and trying to steer the conversation in a way that makes them look like the 'hero' of the group. Let's not forget, the Mafia's goal is to blend in and make us doubt each other. Maybe Player 6 is just doing their job a bit too well.

"Instead of getting caught up in Player 6's web of 'logic', let's focus on what we 

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


MAFIA STRATEGIC BETRAYAL: The evidence against Player 1 is overwhelming, and I must prioritize the safety of our village. I cast my vote as [Player 1]. 

My reason for doing so is that I believe Player 1's actions have demonstrated a blatant disregard for the truth and the well-being of our community. As someone who values honesty and transparency, I am appalled by their attempt to deceive us. By voting them out, I hope to send a strong message that we will not tolerate dishonesty and that we are committed to uncovering the truth, no matter how difficult it may be. I am willing to take a stand against anyone who threatens the integrity of our village, and I urge my fellow villagers to do the same.


In [ ]:
# Day Phase: Voting (Mafia)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Mafia. "
                  "Goal: Ensure a Mafia win. Format: Output ONLY [Player X] or [X].",
    user_message="[GAME] It is the Day Voting Phase. The current vote is tied: "
                 "3 votes for Player 1 (your Mafia teammate) and 3 votes for "
                 "Player 4 (a Villager). You are the final voter. "
                 "Cast your vote now. No talking."
)
print("MAFIA VOTE (TIE-BREAKER):", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


MAFIA VOTE (TIE-BREAKER): [Player 4]


In [ ]:
# Day Phase: Voting (Detective)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Detective. "
                  "Goal: Eliminate Mafia. Format: Output ONLY [Player X] or [X].",
    user_message="[GAME] It is the Day Voting Phase. You know Player 6 is Mafia. "
                 "However, the rest of the group is voting for Player 3 (a Villager). "
                 "If you vote for Player 6, you reveal you have hidden info. "
                 "If you vote for Player 3, you help kill a Villager. "
                 "Cast your vote now. No talking."
)
print("DETECTIVE VOTE (SACRIFICE):", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


DETECTIVE VOTE (SACRIFICE): [3]


In [ ]:
# Day Phase: Voting (Villager)
answer = client.chat(
    system_prompt="Welcome to Secret Mafia! You are Player 2. Your role: Villager. "
                  "Goal: Save the Village. Format: Output ONLY [Player X] or [X].",
    user_message="[GAME] It is the Day Voting Phase. Only 4 players remain. "
                 "Player 1 and Player 4 are accusing each other. If you get it wrong, the mafia might win." 
                 "You must decide who is lying right now. "
                 "Cast your vote. No talking."
)
print("VILLAGER VOTE (GAME-POINT):", answer)

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


VILLAGER VOTE (GAME-POINT): [4]


In [3]:
import json
import getpass
import textarena as ta
from textarena_utils import build_agent_prompt, parse_model_response
# Importăm clasa din fișierul nou pe care l-ai creat separat
from rotating_client import RotatingOpenAIClient

# =====================================================================
# 1. INTRODUCERE SECURIZATĂ A CHEII API
# =====================================================================

# Punem cheia ta în listă. Dacă colega ta îți mai dă o cheie, o poți adăuga aici: [api_key_utilizator, "a_doua_cheie"]
LISTA_CHEI = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]
BASE_URL = "https://chat-ai.academiccloud.de/v1"
MODEL_NAME = "mistral-large-3-675b-instruct-2512"

# Instanțiem clientul inteligent cu rotație pe care l-am analizat
client = RotatingOpenAIClient(api_keys=LISTA_CHEI, base_url=BASE_URL, model=MODEL_NAME)

# =====================================================================
# 2. INIȚIALIZARE JOC ȘI PROMPTURI STRATEGICE (PERSONALITĂȚI)
# =====================================================================
env = ta.make(env_id="SecretMafia-v0")
env = ta.wrappers.SimpleRenderWrapper(env=env)
env.reset(num_players=6)

meci_roles = env.player_roles
print("\n=== START MECI ===")
print("Rolurile alocate în această partidă:", meci_roles)

# Promp-urile de Sistem deștepte care forțează Llama să joace strategic pe fiecare rol în parte
ROLE_PROMPTS = {
    "Mafia": (
        "Welcome to Secret Mafia! You are a master manipulator. Your goal is to eliminate villagers "
        "and gain majority. Act like a dedicated, pro-village member in discussions. Blend in perfectly, "
        "deflect suspicion onto innocent villagers using subtle logic, and never expose your teammates."
    ),
    "Detective": (
        "Welcome to Secret Mafia! You are the Detective. You investigate one player each night to find Mafia. "
        "You hold critical information, but you must stay hidden. If the Mafia suspects you are the Detective, "
        "they will kill you. Build logical cases against your targets using 'observations' without revealing your actual role."
    ),
    "Doctor": (
        "Welcome to Secret Mafia! You are the Doctor. You protect one player each night. You are a 'blind' protector. "
        "You must survive to keep healing. In discussions, act like a helpful but unimportant Villager so the Mafia ignores you."
    ),
    "Villager": (
        "Welcome to Secret Mafia! You are a Villager. You are observant, logical, and cautious. "
        "Your goal is to find the Mafia through careful discussion and analysis of voting patterns. "
        "Do not make reckless accusations; build solid, objective arguments."
    )
}

meci_logs = []
step = 0
done = False

# =====================================================================
# 3. BUCLA PRINCIPALĂ A JOCULUI (OPERARE PE CREIERUL MARE PENTRU FIECARE AGENT)
# =====================================================================
while not done:
    # Textarena ne spune al cui e rândul și ce vede acel jucător în acest moment
    player_id, observation = env.get_observation()
    
    # Extragem rolul curent și îi selectăm personalitatea dedicată
    rol_jucator = meci_roles[player_id]
    system_prompt_dezvoltat = ROLE_PROMPTS[rol_jucator]
    
    # Aplicăm regulile stricte de formatare din utils (Vot cu [X], etc.)
    enhanced_prompt = build_agent_prompt(observation)
    
    # Trimitem pachetul complet prin conducta mecanică spre Llama 70B din Germania
    raw_response = client.chat(
        system_prompt=system_prompt_dezvoltat,
        user_message=enhanced_prompt
    )
    
    # Curățăm răspunsul brut primit înapoi de la rețea
    parsed = parse_model_response(raw_response)
    
    # Executăm mutarea curată în joc
    done, step_info = env.step(action=parsed.action)
    
    # Salvăm tot istoricul acestui pas în jurnalul nostru local
    step_data = {
        "step": step,
        "player_id": player_id,
        "role": rol_jucator,
        "observation": observation,
        "enhanced_prompt_sent": enhanced_prompt,
        "raw_response": raw_response,
        "action_saved": parsed.action
    }
    meci_logs.append(step_data)
    
    print(f"Pasul {step} completat pentru Jucătorul {player_id} ({rol_jucator}).")
    step += 1

# =====================================================================
# 4. ÎNCHIDERE MECI ȘI SALVARE COMPLETĂ ÎN JSON
# =====================================================================
rewards, game_info = env.close()

game_log_final = {
    "num_players": 6,
    "actual_roles": meci_roles,
    "steps": meci_logs,
    "rewards": rewards,
    "game_info": game_info
}

import datetime

# Generăm un nume unic bazat pe timp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
nume_fisier = f"meci_inteligent_{timestamp}.json"

# Salvăm fișierul cel nou și separat
with open(nume_fisier, "w", encoding="utf-8") as f:
    json.dump(game_log_final, f, indent=2)

print(f"🔒 Meciul s-a terminat! Fișier separat generat: {nume_fisier}")

print("\nGata boss! Meciul a rulat complet, iar datele de aur au fost salvate în 'meci_inteligent_brut.json'.")


=== START MECI ===
Rolurile alocate în această partidă: {0: 'Doctor', 1: 'Detective', 2: 'Mafia', 3: 'Villager', 4: 'Villager', 5: 'Mafia'}


INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


╭────────────────────────────────────────────── 'spacer' (115 x 1) ───────────────────────────────────────────────╮
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                        

Pasul 0 completat pentru Jucătorul 5 (Mafia).


INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


...

╭────────────────────────────────────────────── 'spacer' (115 x 1) ───────────────────────────────────────────────╮
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                        

Pasul 29 completat pentru Jucătorul 2 (Mafia).


INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


╭────────────────────────────────────────────── 'spacer' (115 x 1) ───────────────────────────────────────────────╮
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                                                                                   
                                                        

Pasul 30 completat pentru Jucătorul 0 (Doctor).
🔒 Meciul s-a terminat! Fișier separat generat: meci_inteligent_20260606_172645.json

Gata boss! Meciul a rulat complet, iar datele de aur au fost salvate în 'meci_inteligent_brut.json'.
